# Relaxation to the gas temperature

Ions that collide with a neutral gas forget their drift and their temperature.
Maxwell-molecule cross sections make the relaxation rates closed-form, which
tests the collision kinematics directly.

For *Maxwell molecules*, $\sigma = K/g$, the collision frequency $\nu = nK$
does not depend on speed, and the moments relax with closed-form rates.

For isotropic elastic scattering of a particle of mass $m$ on a neutral of mass $M$,
averaging the energy after a collision over the scattering angle and the
neutral's velocity gives

$$\frac{d\langle E\rangle}{dt} = -\frac{2mM}{(m+M)^2}\,\nu\,
\left(\langle E\rangle - \tfrac32 k T_g\right).$$

Energy relaxes fastest for equal masses, at rate $\nu/2$, and slowly for very
different masses. Resonant charge exchange (`backscatter`) hands the ion the
neutral's velocity, so the collided fraction is exactly thermal.

In [ ]:
import matplotlib.pyplot as plt

# light-mode categorical palette, used in this fixed order
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2.0,
        "lines.markersize": 6,
        "legend.frameon": False,
        "figure.figsize": (7.0, 4.2),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.prop_cycle": plt.cycler(color=COLORS),
    }
)

In [ ]:
import numpy as np

import plasmacoll
from plasmacoll.constants import ATOMIC_MASS, BOLTZMANN, ELEMENTARY_CHARGE

M = 4 * ATOMIC_MASS  # the neutral gas
density, K = 1e20, 1e-15  # nu = n K = 1e5 /s
nu = density * K
T_gas = 300.0
gas = plasmacoll.NeutralBackground("gas", density, T_gas, M)


def reactor(kind, mass, temperature, drift=(0.0, 0.0, 0.0), seed=0):
    reduced_mass = mass * M / (mass + M)
    process = plasmacoll.CollisionProcess(
        kind, "gas", plasmacoll.CrossSection.maxwell_molecule(K, reduced_mass)
    )
    mcc = plasmacoll.MonteCarloCollisions(
        {"ion": mass}, [gas], {"ion": [process]}, seed=seed
    )
    ions = plasmacoll.ParticleArrays.maxwellian(
        100_000, mass, temperature, drift=drift, seed=seed + 1
    )
    return plasmacoll.ZeroDReactor(mcc, {"ion": ions})

## Elastic scattering: the mass ratio sets the rate

Three ion masses start hot in a 300 K gas. Points are the simulation and dashed
lines the formula above. The rate is symmetric in $m \leftrightarrow M$: an ion
ten times lighter than the gas relaxes as slowly as one ten times heavier.

In [ ]:
dt = 0.02 / nu
E_gas = 1.5 * BOLTZMANN * T_gas / ELEMENTARY_CHARGE
fig, ax = plt.subplots()
for color, ratio in zip(COLORS, (1.0, 0.25, 0.05), strict=False):
    mass = ratio * M
    history = reactor("elastic", mass, temperature=3000.0).run(dt, 400, every=10)
    t, energy = history.time * nu, history.mean_energy_ev["ion"]
    rate = 2 * mass * M / (mass + M) ** 2
    expected = E_gas + (energy[0] - E_gas) * np.exp(-rate * t)
    ax.plot(t, energy, "o", color=color, label=f"m/M = {ratio:g}")
    ax.plot(t, expected, "--", color=color, linewidth=1.5)
ax.axhline(E_gas, color=MUTED, linewidth=1)
ax.annotate(
    "3/2 kT of the gas",
    (0, E_gas),
    xytext=(0, 4),
    textcoords="offset points",
    color=MUTED,
)
ax.set_xlabel("time × ν")
ax.set_ylabel("mean ion energy (eV)")
ax.set_title("Elastic relaxation, Maxwell molecules (dashed: closed form)", loc="left")
ax.legend();

## Charge exchange: drift and temperature

With resonant charge exchange the fraction $f = e^{-\nu t}$ of ions that have
not collided keeps the initial drift $u_0$ and temperature $T_0$, and the rest is
at the gas temperature. The kinetic temperature about the mean velocity of
this two-population mix is

$$T = T_g + (T_0 - T_g) f + \frac{M u_0^2}{3k} f (1-f),$$

which first *rises* as the beam and the thermalized ions separate in velocity.

In [ ]:
u0, T0 = 2000.0, 3000.0
history = reactor("backscatter", M, T0, drift=(u0, 0.0, 0.0), seed=4).run(
    dt, 250, every=10
)
t = history.time * nu
f = np.exp(-t)
expected = T_gas + (T0 - T_gas) * f + M * u0**2 * f * (1 - f) / (3 * BOLTZMANN)

fig, ax = plt.subplots()
ax.plot(t, history.temperature["ion"], "o", color=COLORS[0], label="simulation")
ax.plot(t, expected, "--", color=INK, linewidth=1.5, label="two-population formula")
ax.set_xlabel("time × ν")
ax.set_ylabel("ion temperature (K)")
ax.set_title("Resonant charge exchange of a drifting ion beam", loc="left")
ax.legend();

The match holds to within a percent or two at $\nu\Delta t = 0.02$. The remaining
difference is the time-step error discussed in
[Ionization and attachment](/plasmacoll/tutorials/03-ionization-attachment/).